In [1]:
import os
import random
import warnings
import time
import operator
from pathlib import Path
from PIL import Image
import cv2
import numpy as np

from joblib import Parallel, delayed, cpu_count
from deap import base, creator, tools, gp
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# ==========================================
# CẤU HÌNH HỆ THỐNG & THAM SỐ
# ==========================================
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

# Đặt seed để dễ dàng tái lập kết quả (Reproducibility)
random.seed(42)
np.random.seed(42)

# Tham số FLGP
POP_SIZE = 100
MAX_GEN = 50
CXPB = 0.8
MUTPB = 0.19
ELITISM = int(0.01 * POP_SIZE)

# THƯ MỤC
THU_MUC_GOC = '/kaggle/input/datasets/xixuhu/office31/Office-31/webcam' 
THU_MUC_MOI = '/kaggle/working/webcam_split_50x50'
DATA_DIR_TRAIN = os.path.join(THU_MUC_MOI, 'train')
DATA_DIR_TEST = os.path.join(THU_MUC_MOI, 'test')

# ==========================================
# 1. HÀM TẢI DỮ LIỆU
# ==========================================
def prepare_train_test_dataset(src_dir, dest_dir, num_train=5, img_size=(50, 50)):
    duong_dan_goc = Path(src_dir)
    thu_muc_train = Path(dest_dir) / 'train'
    thu_muc_test = Path(dest_dir) / 'test'
    
    thu_muc_train.mkdir(parents=True, exist_ok=True)
    thu_muc_test.mkdir(parents=True, exist_ok=True)
    
    dinh_dang_anh = {'.jpg', '.jpeg', '.png', '.bmp'}
    tong_train = tong_test = 0
    
    print(f"Bắt đầu chia tập Train/Test (Train: {num_train} ảnh/class, Resize: {img_size})...", flush=True)
    
    for thu_muc_con in duong_dan_goc.iterdir():
        if not thu_muc_con.is_dir(): continue
            
        (thu_muc_train / thu_muc_con.name).mkdir(parents=True, exist_ok=True)
        (thu_muc_test / thu_muc_con.name).mkdir(parents=True, exist_ok=True)
        
        danh_sach_anh = [f for f in thu_muc_con.glob('*') if f.is_file() and f.suffix.lower() in dinh_dang_anh]
        random.shuffle(danh_sach_anh)
        
        so_luong_train = min(num_train, len(danh_sach_anh))
        anh_train = danh_sach_anh[:so_luong_train]
        anh_test = danh_sach_anh[so_luong_train:]
        
        def process_and_save(img_paths, dest_folder):
            count = 0
            for path in img_paths:
                try:
                    with Image.open(path) as img:
                        img = img.convert('RGB')
                        img_resized = img.resize(img_size, Image.Resampling.LANCZOS)
                        img_resized.save(dest_folder / path.name)
                        count += 1
                except Exception:
                    pass
            return count

        tong_train += process_and_save(anh_train, thu_muc_train / thu_muc_con.name)
        tong_test += process_and_save(anh_test, thu_muc_test / thu_muc_con.name)
        
    print(f"Hoàn thành! Tổng Train: {tong_train} ảnh | Tổng Test: {tong_test} ảnh\n" + "-"*50)

def load_amazon_dataset_raw(data_dir):
    images, labels = [], []
    if not os.path.exists(data_dir): return np.array([]), np.array([])

    class_names = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    class_mapping = {name: idx for idx, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        for img_name in os.listdir(class_dir):
            img = cv2.imread(os.path.join(class_dir, img_name))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append((img / 255.0).astype(np.float32))
                labels.append(class_mapping[class_name])
                
    return np.array(images), np.array(labels, dtype=np.int32)

# ==========================================
# 2. CÁC TOÁN TỬ GP (TỐI ƯU OPENCV)
# ==========================================
def img_add(img1, img2): return np.clip(img1 + img2, 0.0, 1.0)
def img_sub(img1, img2): return np.clip(img1 - img2, 0.0, 1.0)
def img_max(img1, img2): return np.maximum(img1, img2)
def img_relu(img): return np.maximum(0, img)

def img_gaussian_blur(img):
    img_uint8 = np.clip(img * 255, 0, 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(img_uint8, (5, 5), 1.0)
    return (blurred / 255.0).astype(np.float32)

def img_sobel_edges(img):
    img_uint8 = np.clip(img * 255, 0, 255).astype(np.uint8)
    sobelx = cv2.Sobel(img_uint8, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img_uint8, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(sobelx, sobely)
    magnitude = np.clip(magnitude / (np.max(magnitude) + 1e-5), 0, 1)
    return magnitude.astype(np.float32)

if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

pset = gp.PrimitiveSet("MAIN", 1)
pset.renameArguments(ARG0="Image")
pset.addPrimitive(img_add, 2)
pset.addPrimitive(img_sub, 2)
pset.addPrimitive(img_max, 2)
pset.addPrimitive(img_gaussian_blur, 1)
pset.addPrimitive(img_sobel_edges, 1)
pset.addPrimitive(img_relu, 1)

toolbox = base.Toolbox()
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=3)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# ==========================================
# 3. HÀM ĐÁNH GIÁ (CÓ CACHE & 5-FOLD CV)
# ==========================================
FITNESS_CACHE = {}

def gp_transform_data(individual, X_data, pset_local):
    func = gp.compile(expr=individual, pset=pset_local)
    num_features = 6 # Lấy Mean và Std
    X_features = np.zeros((X_data.shape[0], num_features))
    
    for i in range(X_data.shape[0]):
        try:
            processed_img = func(X_data[i])
            if not isinstance(processed_img, np.ndarray) or processed_img.ndim != 3:
                raise ValueError("Lỗi format")
                
            means = np.mean(processed_img, axis=(0, 1))
            stds = np.std(processed_img, axis=(0, 1))
            features = np.concatenate((means, stds))
            X_features[i, :] = np.nan_to_num(features, nan=0.0, posinf=1.0, neginf=0.0)
        except Exception:
            X_features[i, :] = 0.0 
            
    return X_features

def evaluate_singletree_core(ind, X_data, y_data, pset_local):
    tree_str = str(ind)
    
    # Kích hoạt Cache: Bỏ qua nếu cây này đã từng được tính điểm
    if tree_str in FITNESS_CACHE:
        return FITNESS_CACHE[tree_str]

    X_features = gp_transform_data(ind, X_data, pset_local)
    if np.all(X_features == 0):
        FITNESS_CACHE[tree_str] = (0.0,)
        return (0.0,)
        
    # [CẬP NHẬT]: Đánh giá 5-Fold CV với max_iter = 500
    clf = SVC(kernel='linear', max_iter=500, random_state=42)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    accuracies = []
    for train_index, test_index in skf.split(X_features, y_data):
        X_train, X_test = X_features[train_index], X_features[test_index]
        y_train, y_test = y_data[train_index], y_data[test_index]
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        try:
            clf.fit(X_train, y_train)
            accuracies.append(float(np.mean(clf.predict(X_test) == y_test)))
        except:
            accuracies.append(0.0)
            
    # Lấy trung bình 5-Fold nhân 100 ra phần trăm
    final_acc = np.mean(accuracies) * 100
    final_fitness = (final_acc,)
    FITNESS_CACHE[tree_str] = final_fitness
    return final_fitness

def evaluate_wrapper(args):
    ind, X_data, y_data, pset_local = args
    return evaluate_singletree_core(ind, X_data, y_data, pset_local)

# ==========================================
# 4. CẤU HÌNH TIẾN HÓA
# ==========================================
toolbox.register("select_tournament", tools.selTournament, tournsize=5)
toolbox.register("mate", gp.cxOnePoint)
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, pset=pset)
# Bloat Control (Cắt tỉa cây độ sâu tối đa = 8)
toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=8))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=8))

# ==========================================
# 5. VÒNG LẶP TIẾN HÓA
# ==========================================
def main(X_train_raw, y_train_raw, X_test_raw, y_test_raw):
    start_time_total = time.time()
    n_jobs = cpu_count()
    print(f"\n[HỆ THỐNG] Đang chạy FLGP Baseline (Đơn Cây)")
    print(f"[HỆ THỐNG] Phương pháp đánh giá: 5-Fold CV (max_iter=500)")
    print(f"[HỆ THỐNG] Đa luồng: {n_jobs} cores...\n", flush=True)

    pop = toolbox.population(n=POP_SIZE)
    
    print("Đang đánh giá thế hệ đầu tiên...")
    tasks = [(ind, X_train_raw, y_train_raw, pset) for ind in pop]
    results = Parallel(n_jobs=n_jobs, require='sharedmem')(delayed(evaluate_wrapper)(task) for task in tasks)
    
    for ind, res in zip(pop, results):
        ind.fitness.values = res 

    best_acc_global = max(res[0] for res in results)

    for gen in range(MAX_GEN):
        start_time_gen = time.time()
        print(f"\n-- Thế hệ {gen+1}/{MAX_GEN} --", flush=True)
        
        elites = tools.selBest(pop, k=ELITISM)
        elites = list(map(toolbox.clone, elites)) 
        
        num_select = POP_SIZE - ELITISM
        offspring = toolbox.select_tournament(pop, num_select)
        offspring = list(map(toolbox.clone, offspring))

        for i in range(1, len(offspring), 2):
            if random.random() < CXPB:
                toolbox.mate(offspring[i-1], offspring[i])
                del offspring[i-1].fitness.values
                del offspring[i].fitness.values

        for i in range(len(offspring)):
            if random.random() < MUTPB:
                offspring[i], = toolbox.mutate(offspring[i])
                del offspring[i].fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        
        if invalid_ind:
            tasks_gen = [(ind, X_train_raw, y_train_raw, pset) for ind in invalid_ind]
            results_gen = Parallel(n_jobs=n_jobs, require='sharedmem')(delayed(evaluate_wrapper)(task) for task in tasks_gen)
            
            for ind, res in zip(invalid_ind, results_gen):
                ind.fitness.values = res

        pop[:] = elites + offspring
        
        best_ind_gen = tools.selBest(pop, 1)[0]
        if best_ind_gen.fitness.values[0] > best_acc_global:
            best_acc_global = best_ind_gen.fitness.values[0]
            
        print(f"   Độ chính xác tốt nhất thế hệ : {best_ind_gen.fitness.values[0]:.2f}%")
        print(f"   Kỷ lục toàn cục              : {best_acc_global:.2f}%")
        
        time_gen = time.time() - start_time_gen
        print(f"   Thời gian chạy thế hệ        : {time_gen:.2f} giây")

    # TEST CUỐI CÙNG
    print("\n================ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP ================")
    best_ind = tools.selBest(pop, 1)[0]
    
    X_train_feats = gp_transform_data(best_ind, X_train_raw, pset)
    X_test_feats = gp_transform_data(best_ind, X_test_raw, pset)
    
    scaler = StandardScaler()
    X_train_feats = scaler.fit_transform(X_train_feats)
    X_test_feats = scaler.transform(X_test_feats)
    
    # Ở tập test cuối cùng, có thể để max_iter cao hơn hoặc giữ nguyên để nhất quán. Ở đây giữ 2000 để đánh giá đúng năng lực tối đa của cây tốt nhất.
    clf = SVC(kernel='linear', max_iter=2000, random_state=42)
    clf.fit(X_train_feats, y_train_raw)
    test_acc = float(np.mean(clf.predict(X_test_feats) == y_test_raw)) * 100
    
    total_time = time.time() - start_time_total
    hours, rem = divmod(total_time, 3600)
    minutes, seconds = divmod(rem, 60)
    
    print("-" * 50)
    print(f"Độ chính xác Validation (Cao nhất) : {best_ind.fitness.values[0]:.2f}%")
    print(f"ĐỘ CHÍNH XÁC TEST CUỐI CÙNG        : {test_acc:.2f}%")
    print(f"TỔNG THỜI GIAN CHẠY FLGP           : {int(hours)} giờ {int(minutes)} phút {seconds:.2f} giây")
    print("Cây tốt nhất tìm được:")
    print(best_ind)
    print("==================================================================")

# ==========================================
# KHỞI CHẠY CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    if not os.path.exists(THU_MUC_MOI):
        prepare_train_test_dataset(THU_MUC_GOC, THU_MUC_MOI, num_train=5, img_size=(50, 50)) # Chỉnh số lượng tùy ý
    else:
        print(f"Thư mục {THU_MUC_MOI} đã tồn tại. Bỏ qua bước tiền xử lý.")

    print("\n--- ĐANG TẢI TẬP TRAIN ---")
    X_train, y_train = load_amazon_dataset_raw(DATA_DIR_TRAIN)
    print(f"Kích thước tập Train: {X_train.shape}")
    
    print("--- ĐANG TẢI TẬP TEST ---")
    X_test, y_test = load_amazon_dataset_raw(DATA_DIR_TEST)
    print(f"Kích thước tập Test: {X_test.shape}")

    if len(X_train) > 0 and len(X_test) > 0:
        main(X_train, y_train, X_test, y_test)
    else:
        print("Lỗi: Không tìm thấy dữ liệu! Hãy kiểm tra lại đường dẫn THU_MUC_GOC")

Bắt đầu chia tập Train/Test (Train: 5 ảnh/class, Resize: (50, 50))...
Hoàn thành! Tổng Train: 155 ảnh | Tổng Test: 640 ảnh
--------------------------------------------------

--- ĐANG TẢI TẬP TRAIN ---
Kích thước tập Train: (155, 50, 50, 3)
--- ĐANG TẢI TẬP TEST ---
Kích thước tập Test: (640, 50, 50, 3)

[HỆ THỐNG] Đang chạy FLGP Baseline (Đơn Cây)
[HỆ THỐNG] Phương pháp đánh giá: 5-Fold CV (max_iter=500)
[HỆ THỐNG] Đa luồng: 4 cores...

Đang đánh giá thế hệ đầu tiên...

-- Thế hệ 1/50 --
   Độ chính xác tốt nhất thế hệ : 30.97%
   Kỷ lục toàn cục              : 30.97%
   Thời gian chạy thế hệ        : 5.56 giây

-- Thế hệ 2/50 --
   Độ chính xác tốt nhất thế hệ : 30.97%
   Kỷ lục toàn cục              : 30.97%
   Thời gian chạy thế hệ        : 8.14 giây

-- Thế hệ 3/50 --
   Độ chính xác tốt nhất thế hệ : 30.97%
   Kỷ lục toàn cục              : 30.97%
   Thời gian chạy thế hệ        : 6.25 giây

-- Thế hệ 4/50 --
   Độ chính xác tốt nhất thế hệ : 30.97%
   Kỷ lục toàn cục            